# 17 — Mixture of Experts from First Principles

## Goal

A standard Transformer applies the same feed-forward network to every
token representation.

Mixture of Experts changes this assumption.

Instead of using one feed-forward network for every token, the model
contains multiple feed-forward experts and learns a router that decides
which experts should process each token.

This lesson develops MoE from first principles:

1. why Transformer FFNs are natural places for experts,
2. how a router maps token representations to expert scores,
3. how top-k routing creates sparse conditional computation,
4. how tokens are dispatched to experts and gathered back,
5. how total parameters differ from active parameters,
6. why expert imbalance occurs,
7. how load-balancing objectives help,
8. and what capacity constraints mean in real MoE systems.

In [1]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from llmfp.nn import (
    GroupedQueryAttention,
    SwiGLU,
)
from llmfp.utils import count_parameters

## 1. From a Dense FFN to Conditional Computation

A Transformer feed-forward network is token-wise.

For a residual-stream tensor

$$
X\in\mathbb{R}^{B\times T\times C},
$$

the same function

$$
f:\mathbb{R}^C\rightarrow\mathbb{R}^C
$$

is applied independently to every token representation:

$$
y_{b,t}=f(x_{b,t}).
$$

Tokens interact through attention, but the feed-forward network does not
mix sequence positions.

This raises a natural question:

> Why must every token use the same feed-forward parameters?

In [2]:
embedding_dim: int = 256
hidden_dim: int = 768

dense_ffn = SwiGLU(
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
)

print(f"dense FFN parameters: {count_parameters(dense_ffn):,}")

dense FFN parameters: 589,824


## 2. The Router

For $E$ experts, the router maps one token representation

$$
x\in\mathbb{R}^{C}
$$

to $E$ routing logits:

$$
r
=
W_r x,
$$

where

$$
W_r\in\mathbb{R}^{E\times C}.
$$

Therefore,

$$
r\in\mathbb{R}^{E}.
$$

Applying softmax gives routing probabilities:

$$
p(e\mid x)
=
\operatorname{softmax}(r)_e.
$$

The router is therefore a small learned classifier over experts.

In [3]:
num_experts: int = 4
embedding_dim: int = 8

router = nn.Linear(
    embedding_dim,
    num_experts,
    bias=False,
)

token: torch.Tensor = torch.randn(embedding_dim)

router_logits: torch.Tensor = router(token)

router_probabilities: torch.Tensor = F.softmax(
    router_logits,
    dim=-1,
)

print(
    "token:",
    token.shape,
)

print(
    "router logits:",
    router_logits.shape,
)

print(
    "router probabilities:",
    router_probabilities,
)

print(
    "sum:",
    router_probabilities.sum(),
)

token: torch.Size([8])
router logits: torch.Size([4])
router probabilities: tensor([0.1091, 0.5654, 0.0852, 0.2403], grad_fn=<SoftmaxBackward0>)
sum: tensor(1., grad_fn=<SumBackward0>)


In [4]:
selected_expert: torch.Tensor = torch.argmax(router_probabilities, dim=-1)

print("selected expert:", selected_expert.item())

selected expert: 1


## 3. Manual Top-1 Dispatch and Gather

A token-level MoE routes each token representation independently.

For an input tensor

$$
X\in\mathbb{R}^{B\times T\times C},
$$

there are

$$
N=BT
$$

token representations.

We can therefore flatten the batch and sequence dimensions:

$$
(B,T,C)
\rightarrow
(N,C).
$$

The router produces one expert distribution for every token:

$$
R
\in
\mathbb{R}^{N\times E},
$$

where $E$ is the number of experts.

For top-1 routing, each token selects one expert:

$$
e_i
=
\arg\max_e R_{i,e}.
$$

The implementation then performs two structural operations:

<pre>
dispatch
tokens → assigned experts

gather
expert outputs → original token positions
</pre>

The external Transformer interface remains unchanged:

$$
(B,T,C)
\rightarrow
(B,T,C).
$$

In [5]:
batch_size: int = 2
sequence_length: int = 3
embedding_dim: int = 8
hidden_dim: int = 16
num_experts: int = 4

x: torch.Tensor = torch.randn(
    batch_size,
    sequence_length,
    embedding_dim,
)

router = nn.Linear(
    embedding_dim,
    num_experts,
    bias=False,
)

experts = nn.ModuleList(
    [
        SwiGLU(
            embedding_dim=embedding_dim,
            hidden_dim=hidden_dim,
        )
        for _ in range(num_experts)
    ]
)

print("input:", x.shape)

input: torch.Size([2, 3, 8])


In [6]:
x_flat: torch.Tensor = x.reshape(
    batch_size * sequence_length,
    embedding_dim,
)  # flatten all seqs to one dim.

print("flattened:", x_flat.shape)

flattened: torch.Size([6, 8])


In [7]:
router_logits: torch.Tensor = router(x_flat)

router_probabilities: torch.Tensor = F.softmax(router_logits, dim=-1)

expert_assignments: torch.Tensor = torch.argmax(router_probabilities, dim=-1)

print("router logits:", router_logits.shape)

print("router probabilities:", router_probabilities.shape)

print("assignments:", expert_assignments)

router logits: torch.Size([6, 4])
router probabilities: torch.Size([6, 4])
assignments: tensor([1, 0, 3, 2, 1, 0])


In [8]:
expert_token_indices: list[list[int]] = [[] for _ in range(num_experts)]

num_tokens: int = x_flat.shape[0]

for token_index in range(num_tokens):
    expert_index: int = int(expert_assignments[token_index].item())
    expert_token_indices[expert_index].append(token_index)

for expert_index, token_indices in enumerate(expert_token_indices):
    print(f"expert {expert_index}:", token_indices)

expert 0: [1, 5]
expert 1: [0, 4]
expert 2: [3]
expert 3: [2]


In [9]:
expert_index: int = 2

token_indices: list[int] = expert_token_indices[expert_index]

index_tensor: torch.Tensor = torch.tensor(
    token_indices, dtype=torch.long, device=x.device
)
expert_inputs: torch.Tensor = x_flat[index_tensor]

print(
    "expert inputs:",
    expert_inputs.shape,
)

print("expert inputs:", expert_inputs)

expert inputs: torch.Size([1, 8])
expert inputs: tensor([[ 0.1810, -0.3619, -1.1499, -0.4267,  0.3782, -0.6765, -0.9763, -1.5092]])


In [10]:
expert_outputs: torch.Tensor = experts[expert_index](expert_inputs)

print(
    "expert outputs:",
    expert_outputs.shape,
)

expert outputs: torch.Size([1, 8])


In [11]:
def manual_top1_moe(
    x: torch.Tensor, router: nn.Linear, experts: nn.ModuleList
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Apply a manually dispatched top-1 mixture of experts.

    Args:
        x: Token representations with shape `(B, T, C)`.
        router: Linear router mapping `C -> E`.
        experts: Expert feed-forward modules.

    Returns:
        A tuple containing:
        - MoE output with shape `(B, T, C)`,
        - router probabilities with shape `(B*T, E)`,
        - selected expert indices with shape `(B*T)`.
    """
    if x.ndim != 3:
        raise ValueError("x must have the shape like (B, T, C)")

    batch_size, sequence_length, embedding_dim = x.shape
    num_tokens: int = batch_size * sequence_length
    num_experts: int = len(experts)

    x_flat: torch.Tensor = x.reshape(num_tokens, embedding_dim)

    router_logits: torch.Tensor = router(x_flat)

    router_probabilities: torch.Tensor = F.softmax(router_logits, dim=-1)

    expert_assignments: torch.Tensor = torch.argmax(
        router_probabilities, dim=-1
    )

    # Record which original token positions belong to each expert.
    expert_token_indices: list[list[int]] = [[] for _ in range(num_experts)]

    for token_index in range(num_tokens):
        expert_index: int = int(expert_assignments[token_index].item())
        expert_token_indices[expert_index].append(token_index)

    # Each slot will eventually contain the output belonging to the correspoding original token position.
    output_slots: list[torch.Tensor | None] = [None for _ in range(num_tokens)]

    for expert_index, token_indices in enumerate(expert_token_indices):
        if not token_indices:
            continue

        index_tensor: torch.Tensor = torch.tensor(
            token_indices, dtype=torch.long, device=x.device
        )

        expert_inputs: torch.Tensor = x_flat[index_tensor]

        expert_outputs: torch.Tensor = experts[expert_index](expert_inputs)

        # Keep the selected router probability as a differentiable gate.

        routing_weights: torch.Tensor = router_probabilities[
            index_tensor, expert_index
        ].unsqueeze(-1)

        weighted_outputs: torch.Tensor = routing_weights * expert_outputs

        # Gather each result back into its original token position.

        for local_index, token_index in enumerate(token_indices):
            output_slots[token_index] = weighted_outputs[local_index]

    if any(output is None for output in output_slots):
        raise RuntimeError(
            "Every token must receive exactly one epxert output."
        )

    output_flat: torch.Tensor = torch.stack(
        [output for output in output_slots if output is not None], dim=0
    )

    output: torch.Tensor = output_flat.reshape(
        batch_size, sequence_length, embedding_dim
    )

    return output, router_probabilities, expert_assignments

In [12]:
moe_output, routing_probabilities, assignments = manual_top1_moe(
    x,
    router,
    experts,
)

print(
    "input:",
    x.shape,
)

print(
    "output:",
    moe_output.shape,
)

print(
    "routing probabilities:",
    routing_probabilities.shape,
)

print(
    "assignments:",
    assignments.shape,
)

input: torch.Size([2, 3, 8])
output: torch.Size([2, 3, 8])
routing probabilities: torch.Size([6, 4])
assignments: torch.Size([6])


In [13]:
for expert_index in range(num_experts):
    count: int = 0

    for token_index in range(assignments.shape[0]):
        if int(assignments[token_index].item()) == expert_index:
            count += 1

    print(f"expert {expert_index}: {count} tokens")

expert 0: 2 tokens
expert 1: 2 tokens
expert 2: 1 tokens
expert 3: 1 tokens


In [14]:
router.zero_grad()

for expert in experts:
    expert.zero_grad()

moe_output, _, assignments = manual_top1_moe(
    x,
    router,
    experts,
)

loss: torch.Tensor = moe_output.square().mean()

loss.backward()

print(
    "router gradient norm:",
    torch.linalg.vector_norm(router.weight.grad).item(),
)

for expert_index, expert in enumerate(experts):
    first_parameter = next(expert.parameters())

    if first_parameter.grad is None:
        print(f"expert {expert_index}: no gradient")
    else:
        print(
            f"expert {expert_index}:",
            torch.linalg.vector_norm(first_parameter.grad).item(),
        )

router gradient norm: 0.010003392584621906
expert 0: 0.00974960345774889
expert 1: 0.0046480679884552956
expert 2: 0.00025301682762801647
expert 3: 0.0005141086876392365


### Routing Allocates Both Compute and Learning Signal

In a dense FFN, every token passes through the same parameters.

In a sparse MoE, only selected experts process a token.

Therefore routing determines both:

<pre>
which parameters execute
        +
which parameters receive task gradients
</pre>

An expert that receives no tokens contributes nothing to the current
task loss and receives no task gradient from that batch.

This creates a feedback problem:

<pre>
router prefers one expert
        ↓
that expert receives more data
        ↓
that expert learns faster
        ↓
router may prefer it even more


This motivates explicit mechanisms for balancing expert utilization.

## 4. Top-k Routing

Top-1 routing sends each token to exactly one expert.

More generally, a sparse MoE can select

$$
k
$$

experts for each token.

For top-$k$ routing, token $i$ selects a set of experts

$$
\mathcal{E}_i
=
\operatorname{TopK}(p_i, k),
$$

where

$$
p_i
=
\operatorname{softmax}(r_i)
$$

is the router distribution.

The MoE output is then a weighted combination of the selected expert
outputs:

$$
y_i
=
\sum_{e\in\mathcal{E}_i}
g_{i,e} f_e(x_i).
$$

Only the selected experts are executed for that token.

### Renormalizing the Selected Experts

After selecting the top-$k$ experts, we renormalize their routing weights:

$$
g_{i,e}
=
\frac{
p_{i,e}
}{
\sum_{e'\in\mathcal{E}_i} p_{i,e'}
}.
$$

Therefore,

$$
\sum_{e\in\mathcal{E}_i}
g_{i,e}
=
1.
$$

The router first decides which experts are active, then the selected
weights determine how strongly their outputs contribute.

In [15]:
top_k: int = 2

router_logits: torch.Tensor = router(x_flat)

router_probabilities: torch.Tensor = F.softmax(
    router_logits,
    dim=-1,
)

topk_probabilities, topk_indices = torch.topk(
    router_probabilities,
    k=top_k,
    dim=-1,
)

print(
    "router probabilities:",
    router_probabilities.shape,
)

print(
    "top-k probabilities:",
    topk_probabilities.shape,
)

print(
    "top-k indices:",
    topk_indices.shape,
)

router probabilities: torch.Size([6, 4])
top-k probabilities: torch.Size([6, 2])
top-k indices: torch.Size([6, 2])


In [16]:
topk_weights: torch.Tensor = topk_probabilities / topk_probabilities.sum(
    dim=-1,
    keepdim=True,
)

print(topk_weights.sum(dim=-1))

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)


In [17]:
token_index: int = 0

print(
    "all expert probabilities:",
    router_probabilities[token_index],
)

print(
    "selected experts:",
    topk_indices[token_index],
)

print(
    "selected weights:",
    topk_weights[token_index],
)

all expert probabilities: tensor([0.2287, 0.3796, 0.1396, 0.2521], grad_fn=<SelectBackward0>)
selected experts: tensor([1, 3])
selected weights: tensor([0.6010, 0.3990], grad_fn=<SelectBackward0>)


In [18]:
expert_routes: list[list[tuple[int, int]]] = [[] for _ in range(num_experts)]

num_tokens: int = x_flat.shape[0]

for token_index in range(num_tokens):
    for route_position in range(top_k):
        expert_index: int = int(
            topk_indices[
                token_index,
                route_position,
            ].item()
        )

        expert_routes[expert_index].append(
            (
                token_index,
                route_position,
            )
        )

In [19]:
for expert_index, routes in enumerate(expert_routes):
    print(
        f"expert {expert_index}:",
        routes,
    )

expert 0: [(1, 0), (5, 0)]
expert 1: [(0, 0), (2, 1), (4, 0), (5, 1)]
expert 2: [(1, 1), (3, 0), (4, 1)]
expert 3: [(0, 1), (2, 0), (3, 1)]


In [20]:
def manual_topk_moe(
    x: torch.Tensor,
    router: nn.Linear,
    experts: nn.ModuleList,
    top_k: int,
) -> tuple[
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
]:
    """Apply manually dispatched sparse top-k mixture of experts.

    Args:
        x: Token representations with shape `(B, T, C)`.
        router: Linear router mapping `C -> E`.
        experts: Expert feed-forward modules.
        top_k: Number of experts selected for every token.

    Returns:
        A tuple containing:
        - MoE output with shape `(B, T, C)`,
        - full router probabilities with shape `(B*T, E)`,
        - selected expert indices with shape `(B*T, K)`,
        - normalized selected weights with shape `(B*T, K)`.
    """
    if x.ndim != 3:
        raise ValueError("x must have shape (B, T, C).")

    batch_size, sequence_length, embedding_dim = x.shape

    num_experts: int = len(experts)

    if not 1 <= top_k <= num_experts:
        raise ValueError("top_k must be between 1 and num_experts.")

    num_tokens: int = batch_size * sequence_length

    x_flat: torch.Tensor = x.reshape(
        num_tokens,
        embedding_dim,
    )

    router_logits: torch.Tensor = router(x_flat)

    router_probabilities: torch.Tensor = F.softmax(
        router_logits,
        dim=-1,
    )

    (
        topk_probabilities,
        topk_indices,
    ) = torch.topk(
        router_probabilities,
        k=top_k,
        dim=-1,
    )

    # Renormalize only among selected experts.
    topk_weights: torch.Tensor = topk_probabilities / topk_probabilities.sum(
        dim=-1,
        keepdim=True,
    )

    # Each expert stores:
    #
    # (original token index, selected-route position)
    expert_routes: list[list[tuple[int, int]]] = [
        [] for _ in range(num_experts)
    ]

    for token_index in range(num_tokens):
        for route_position in range(top_k):
            expert_index: int = int(
                topk_indices[
                    token_index,
                    route_position,
                ].item()
            )

            expert_routes[expert_index].append(
                (
                    token_index,
                    route_position,
                )
            )

    # A token may receive multiple expert contributions.
    token_contributions: list[list[torch.Tensor]] = [
        [] for _ in range(num_tokens)
    ]

    for expert_index, routes in enumerate(expert_routes):
        if not routes:
            continue

        token_indices: list[int] = [token_index for token_index, _ in routes]

        route_positions: list[int] = [
            route_position for _, route_position in routes
        ]

        token_index_tensor = torch.tensor(
            token_indices,
            dtype=torch.long,
            device=x.device,
        )

        route_position_tensor = torch.tensor(
            route_positions,
            dtype=torch.long,
            device=x.device,
        )

        expert_inputs: torch.Tensor = x_flat[token_index_tensor]

        expert_outputs: torch.Tensor = experts[expert_index](expert_inputs)

        routing_weights: torch.Tensor = topk_weights[
            token_index_tensor,
            route_position_tensor,
        ].unsqueeze(-1)

        weighted_outputs: torch.Tensor = routing_weights * expert_outputs

        for local_index, token_index in enumerate(token_indices):
            token_contributions[token_index].append(
                weighted_outputs[local_index]
            )

    output_tokens: list[torch.Tensor] = []

    for contributions in token_contributions:
        if len(contributions) != top_k:
            raise RuntimeError(
                "Each token must receive exactly top_k expert contributions."
            )

        token_output: torch.Tensor = torch.stack(
            contributions,
            dim=0,
        ).sum(dim=0)

        output_tokens.append(token_output)

    output_flat: torch.Tensor = torch.stack(
        output_tokens,
        dim=0,
    )

    output: torch.Tensor = output_flat.reshape(
        batch_size,
        sequence_length,
        embedding_dim,
    )

    return (
        output,
        router_probabilities,
        topk_indices,
        topk_weights,
    )

In [21]:
(
    moe_output,
    router_probabilities,
    selected_experts,
    selected_weights,
) = manual_topk_moe(
    x,
    router,
    experts,
    top_k=2,
)

print(
    "input:",
    x.shape,
)

print(
    "output:",
    moe_output.shape,
)

print(
    "selected experts:",
    selected_experts.shape,
)

print(
    "selected weights:",
    selected_weights.shape,
)

input: torch.Size([2, 3, 8])
output: torch.Size([2, 3, 8])
selected experts: torch.Size([6, 2])
selected weights: torch.Size([6, 2])


In [22]:
def expert_utilization(
    selected_experts: torch.Tensor,
    num_experts: int,
) -> torch.Tensor:
    """Measure the fraction of routing assignments sent to each expert.

    Args:
        selected_experts: Expert IDs with shape `(N, K)`.
        num_experts: Total number of experts.

    Returns:
        Utilization fractions with shape `(E)`.
    """
    counts: list[float] = []

    total_assignments: int = selected_experts.numel()

    for expert_index in range(num_experts):
        count: int = int((selected_experts == expert_index).sum().item())

        counts.append(count / total_assignments)

    return torch.tensor(
        counts,
        dtype=torch.float32,
        device=selected_experts.device,
    )

In [23]:
utilization: torch.Tensor = expert_utilization(
    selected_experts,
    num_experts=num_experts,
)

print(utilization)
print(utilization.sum())

tensor([0.1667, 0.3333, 0.2500, 0.2500])
tensor(1.)


### Top-k Routing Introduces a New Trade-off

Increasing $k$ allows every token to use more expert capacity:

$$
y_i
=
\sum_{e\in\mathcal{E}_i}
g_{i,e} f_e(x_i).
$$

But it also increases expert computation.

Therefore, MoE separates two quantities:

$$
E
=
\text{number of available experts},
$$

and

$$
k
=
\text{number of active experts per token}.
$$

The ratio between available and active experts is one of the central
design choices in sparse MoE systems.

## 5. Load Balancing the Experts

A sparse MoE can collapse toward a small subset of experts.

For example, with four experts, routing assignments might look like

$$
f
=
[0.70,\ 0.20,\ 0.10,\ 0.00].
$$

The language-modeling objective does not directly require balanced expert
usage.

If one expert happens to produce good predictions, the task loss may be
perfectly happy to route more and more tokens to that expert.

This creates both optimization and systems problems:

- overloaded experts receive too many tokens,
- unused experts receive little or no task gradient,
- parameter capacity is wasted,
- and hardware workload becomes imbalanced.

We therefore need an additional signal that encourages the router to use
experts more evenly.

### Hard Routing Counts Are Not Enough

The actual routing fraction

$$
f_e
$$

depends on discrete top-k selections.

Small changes in router logits often leave the selected expert IDs
unchanged.

Therefore, routing counts alone do not provide a useful ordinary
gradient path back to the router.

The soft router probabilities remain differentiable, however.

In [24]:
mean_router_probability: torch.Tensor = router_probabilities.mean(dim=0)

print(mean_router_probability)

print(mean_router_probability.sum())

tensor([0.2654, 0.2706, 0.2554, 0.2086], grad_fn=<MeanBackward1>)
tensor(1.0000, grad_fn=<SumBackward0>)


In [25]:
utilization: torch.Tensor = expert_utilization(
    selected_experts,
    num_experts=num_experts,
)

print(utilization)

print(utilization.sum())

tensor([0.1667, 0.3333, 0.2500, 0.2500])
tensor(1.)


In [26]:
def load_balancing_loss(
    router_probabilities: torch.Tensor,
    selected_experts: torch.Tensor,
    num_experts: int,
) -> torch.Tensor:
    """Encourage balanced expert routing.

    Args:
        router_probabilities: Full router probabilities with shape
            `(N, E)`.
        selected_experts: Selected expert IDs with shape `(N, K)`.
        num_experts: Total number of experts.

    Returns:
        Scalar auxiliary load-balancing loss.
    """
    if router_probabilities.ndim != 2:
        raise ValueError("router_probabilities must have shape (N, E)")

    if selected_experts.ndim != 2:
        raise ValueError("selected_experts must have shape (N, K)")

    if router_probabilities.shape[1] != num_experts:
        raise ValueError("router probability width must match num_experts")

    utilization: torch.Tensor = expert_utilization(
        selected_experts, num_experts=num_experts
    )

    mean_router_probability: torch.Tensor = router_probabilities.mean(dim=0)

    return num_experts * torch.sum(utilization * mean_router_probability)


In [27]:
(
    moe_output,
    router_probabilities,
    selected_experts,
    selected_weights,
) = manual_topk_moe(
    x,
    router,
    experts,
    top_k=2,
)

balance_loss: torch.Tensor = load_balancing_loss(
    router_probabilities,
    selected_experts,
    num_experts=num_experts,
)

print(
    "load balancing loss:",
    balance_loss.item(),
)

load balancing loss: 1.001739740371704


In [28]:
for expert_index, expert in enumerate(experts):
    parameter = next(expert.parameters())

    print(
        expert_index,
        parameter.grad,
    )

0 tensor([[-1.1324e-04,  3.5135e-04, -1.5303e-04,  1.3500e-04, -3.7822e-05,
         -2.7864e-04,  1.3416e-04, -1.6806e-04],
        [ 1.6798e-05,  1.8014e-06,  1.3009e-05,  8.3998e-06, -8.9383e-06,
          1.3622e-05,  2.7570e-06,  2.9342e-05],
        [-1.0779e-04,  3.3410e-04, -1.4560e-04,  1.2832e-04, -3.5908e-05,
         -2.6506e-04,  1.2756e-04, -1.6001e-04],
        [ 5.0984e-04, -1.9918e-03,  7.6267e-04, -8.2390e-04,  2.8088e-04,
          1.4652e-03, -7.7629e-04,  7.2319e-04],
        [ 1.0321e-04, -4.0467e-04,  1.5465e-04, -1.6756e-04,  5.7255e-05,
          2.9736e-04, -1.5776e-04,  1.4627e-04],
        [ 3.1710e-04, -1.1857e-03,  4.6481e-04, -4.8445e-04,  1.6037e-04,
          8.8404e-04, -4.6052e-04,  4.5414e-04],
        [-1.2910e-04,  4.9982e-04, -1.9231e-04,  2.0623e-04, -6.9897e-05,
         -3.6869e-04,  1.9466e-04, -1.8350e-04],
        [ 1.5928e-03, -5.9797e-03,  2.3390e-03, -2.4459e-03,  8.1195e-04,
          4.4528e-03, -2.3232e-03,  2.2792e-03],
        [ 9.76

### Task Loss and Routing Loss Solve Different Problems

The language-modeling loss asks:

> Which routing decisions improve prediction?

The balancing loss asks:

> Is computation distributed too unevenly across experts?

The combined objective is

$$
\mathcal{L}
=
\mathcal{L}_{\text{task}}
+
\lambda
\mathcal{L}_{\text{balance}}.
$$

The coefficient $\lambda$ expresses a trade-off.

If it is too small, routing may collapse.

If it is too large, the router may be pushed toward uniform utilization
even when stronger specialization would help the task.

## 6. Expert Capacity and Overflow

Load balancing encourages experts to receive similar amounts of work.

However, an auxiliary loss does not guarantee perfectly balanced routing
for every batch.

Real sparse MoE systems therefore often impose an explicit capacity on
each expert.

Suppose a batch contains

$$
N
$$

tokens, each token selects

$$
k
$$

experts, and there are

$$
E
$$

experts.

The total number of token-expert routes is

$$
Nk.
$$

If routing were perfectly balanced, each expert would receive

$$
\frac{Nk}{E}
$$

routes on average.

A simple expert-capacity rule is

$$
C_{\text{expert}}
=
\left\lceil
\alpha
\frac{Nk}{E}
\right\rceil,
$$

where

$$
\alpha
$$

is the capacity factor.

In [29]:
def expert_capacity(
    num_tokens: int,
    top_k: int,
    num_experts: int,
    capacity_factor: float,
) -> int:
    """Compute the maximum number of routes allowed per expert.

    Args:
        num_tokens: Number of token representations.
        top_k: Number of selected experts per token.
        num_experts: Total number of experts.
        capacity_factor: Extra capacity relative to perfectly balanced load.

    Returns:
        Maximum number of token-expert routes per expert.
    """
    if num_tokens <= 0:
        raise ValueError("num_tokens must be positive.")

    if not 1 <= top_k <= num_experts:
        raise ValueError("top_k must be between 1 and num_experts.")

    if capacity_factor <= 0.0:
        raise ValueError("capacity_factor must be positive.")

    average_routes_per_expert: float = num_tokens * top_k / num_experts

    return math.ceil(capacity_factor * average_routes_per_expert)

In [30]:
capacity: int = expert_capacity(
    num_tokens=1024,
    top_k=2,
    num_experts=8,
    capacity_factor=1.25,
)

print(capacity)

320


In [31]:
capacity: int = expert_capacity(
    num_tokens=selected_experts.shape[0],
    top_k=selected_experts.shape[1],
    num_experts=num_experts,
    capacity_factor=1.0,
)

print(
    "expert capacity:",
    capacity,
)

for expert_index in range(num_experts):
    route_count: int = int((selected_experts == expert_index).sum().item())

    overflow: int = max(
        0,
        route_count - capacity,
    )

    print(f"expert {expert_index}: routes={route_count}, overflow={overflow}")

expert capacity: 3
expert 0: routes=2, overflow=0
expert 1: routes=4, overflow=1
expert 2: routes=3, overflow=0
expert 3: routes=3, overflow=0


### Capacity Introduces a Hard Systems Constraint

A balancing loss encourages the router to distribute work.

Expert capacity limits how much work an expert is actually allowed to
receive.

For $N$ tokens, top-$k$ routing, $E$ experts, and capacity factor
$\alpha$, a simple capacity rule is

$$
C_{\text{expert}}
=
\left\lceil
\alpha
\frac{Nk}{E}
\right\rceil.
$$

This introduces a distinction between

<pre>
desired routing
        ↓
top-k assignments

and

actual executable routing
        ↓
capacity-constrained assignments
</pre>

Sparse MoE therefore creates a resource-allocation problem in addition to
the usual optimization problem.

## 7. Building a Sparse MoE Layer

We now combine the mechanisms developed so far into one module:

<pre>
token representations
        ↓
router
        ↓
top-k selection
        ↓
optional capacity filtering
        ↓
dispatch
        ↓
expert FFNs
        ↓
weighted gather
        ↓
token representations
</pre>

The external interface remains

$$
(B,T,C)
\rightarrow
(B,T,C).
$$

The MoE layer also exposes routing information and an auxiliary
load-balancing loss for training and inspection.

In [32]:
class SparseMoE(nn.Module):
    """Sparse top-k mixture of SwiGLU experts.

    Args:
        embedding_dim: Width of the residual stream.
        hidden_dim: Hidden width of every expert.
        num_experts: Number of available experts.
        top_k: Number of experts activated per token.
        capacity_factor: Optional expert-capacity multiplier.
            If `None`, no routes are dropped.
    """

    def __init__(
        self,
        embedding_dim: int,
        hidden_dim: int,
        num_experts: int,
        top_k: int,
        capacity_factor: float | None = None,
    ) -> None:
        super().__init__()

        if embedding_dim <= 0:
            raise ValueError("embedding_dim must be positive.")

        if hidden_dim <= 0:
            raise ValueError("hidden_dim must be positive.")

        if num_experts <= 0:
            raise ValueError("num_experts must be positive.")

        if not 1 <= top_k <= num_experts:
            raise ValueError("top_k must be between 1 and num_experts.")

        if capacity_factor is not None and capacity_factor <= 0.0:
            raise ValueError("capacity_factor must be positive.")

        self.embedding_dim: int = embedding_dim
        self.hidden_dim: int = hidden_dim
        self.num_experts: int = num_experts
        self.top_k: int = top_k
        self.capacity_factor: float | None = capacity_factor

        self.router = nn.Linear(
            embedding_dim,
            num_experts,
            bias=False,
        )

        self.experts = nn.ModuleList(
            [
                SwiGLU(
                    embedding_dim=embedding_dim,
                    hidden_dim=hidden_dim,
                )
                for _ in range(num_experts)
            ]
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        """Apply sparse MoE and return only the token updates."""
        output, _, _ = self.forward_with_auxiliary(x)

        return output

    def forward_with_auxiliary(
        self,
        x: torch.Tensor,
    ) -> tuple[
        torch.Tensor,
        torch.Tensor,
        dict[str, torch.Tensor],
    ]:
        """Apply sparse MoE and expose routing information.

        Args:
            x: Token representations with shape `(B, T, C)`.

        Returns:
            A tuple containing:
            - MoE output with shape `(B, T, C)`,
            - scalar load-balancing loss,
            - routing diagnostics.
        """
        if x.ndim != 3:
            raise ValueError("x must have shape (B, T, C).")

        (
            batch_size,
            sequence_length,
            embedding_dim,
        ) = x.shape

        if embedding_dim != self.embedding_dim:
            raise ValueError("Input width must match embedding_dim.")

        num_tokens: int = batch_size * sequence_length

        x_flat: torch.Tensor = x.reshape(
            num_tokens,
            embedding_dim,
        )

        # ------------------------------------------------------------
        # 1. Router
        # ------------------------------------------------------------

        router_logits: torch.Tensor = self.router(x_flat)

        router_probabilities: torch.Tensor = F.softmax(
            router_logits,
            dim=-1,
        )

        (
            topk_probabilities,
            topk_indices,
        ) = torch.topk(
            router_probabilities,
            k=self.top_k,
            dim=-1,
        )

        # Renormalize probability mass over only the active experts.
        topk_weights: torch.Tensor = (
            topk_probabilities
            / topk_probabilities.sum(
                dim=-1,
                keepdim=True,
            )
        )

        # ------------------------------------------------------------
        # 2. Auxiliary balancing objective
        # ------------------------------------------------------------

        balance_loss: torch.Tensor = load_balancing_loss(
            router_probabilities,
            topk_indices,
            num_experts=self.num_experts,
        )

        utilization: torch.Tensor = expert_utilization(
            topk_indices,
            num_experts=self.num_experts,
        )

        # ------------------------------------------------------------
        # 3. Capacity filtering
        # ------------------------------------------------------------

        accepted_route_mask: torch.Tensor = torch.ones_like(
            topk_indices,
            dtype=torch.bool,
        )

        if self.capacity_factor is not None:
            capacity: int = expert_capacity(
                num_tokens=num_tokens,
                top_k=self.top_k,
                num_experts=self.num_experts,
                capacity_factor=self.capacity_factor,
            )

            accepted_route_mask = torch.zeros_like(
                topk_indices,
                dtype=torch.bool,
            )

            for expert_index in range(self.num_experts):
                routes: list[tuple[int, int, float]] = []

                for token_index in range(num_tokens):
                    for route_position in range(self.top_k):
                        if (
                            int(
                                topk_indices[
                                    token_index,
                                    route_position,
                                ].item()
                            )
                            != expert_index
                        ):
                            continue

                        route_weight: float = float(
                            topk_weights[
                                token_index,
                                route_position,
                            ]
                            .detach()
                            .item()
                        )

                        routes.append(
                            (
                                token_index,
                                route_position,
                                route_weight,
                            )
                        )

                # Capacity selection itself is discrete, so using detached
                # routing weights for this ranking is intentional.
                routes.sort(
                    key=lambda route: route[2],
                    reverse=True,
                )

                for (
                    token_index,
                    route_position,
                    _,
                ) in routes[:capacity]:
                    accepted_route_mask[
                        token_index,
                        route_position,
                    ] = True

        # ------------------------------------------------------------
        # 4. Build accepted dispatch plan
        # ------------------------------------------------------------

        expert_routes: list[list[tuple[int, int]]] = [
            [] for _ in range(self.num_experts)
        ]

        for token_index in range(num_tokens):
            for route_position in range(self.top_k):
                if not bool(
                    accepted_route_mask[
                        token_index,
                        route_position,
                    ].item()
                ):
                    continue

                expert_index: int = int(
                    topk_indices[
                        token_index,
                        route_position,
                    ].item()
                )

                expert_routes[expert_index].append(
                    (
                        token_index,
                        route_position,
                    )
                )

        # ------------------------------------------------------------
        # 5. Dispatch, expert computation, and gather
        # ------------------------------------------------------------

        token_contributions: list[list[torch.Tensor]] = [
            [] for _ in range(num_tokens)
        ]

        for expert_index, routes in enumerate(expert_routes):
            if not routes:
                continue

            token_indices: list[int] = [
                token_index for token_index, _ in routes
            ]

            route_positions: list[int] = [
                route_position for _, route_position in routes
            ]

            token_index_tensor = torch.tensor(
                token_indices,
                dtype=torch.long,
                device=x.device,
            )

            route_position_tensor = torch.tensor(
                route_positions,
                dtype=torch.long,
                device=x.device,
            )

            expert_inputs: torch.Tensor = x_flat[token_index_tensor]

            expert_outputs: torch.Tensor = self.experts[expert_index](
                expert_inputs
            )

            routing_weights: torch.Tensor = topk_weights[
                token_index_tensor,
                route_position_tensor,
            ].unsqueeze(-1)

            weighted_outputs: torch.Tensor = routing_weights * expert_outputs

            for (
                local_index,
                token_index,
            ) in enumerate(token_indices):
                token_contributions[token_index].append(
                    weighted_outputs[local_index]
                )

        # ------------------------------------------------------------
        # 6. Restore original token order
        # ------------------------------------------------------------

        output_tokens: list[torch.Tensor] = []

        for token_index, contributions in enumerate(token_contributions):
            if not contributions:
                # If every route for a token overflowed, this MoE contributes
                # a zero FFN update. The outer Transformer residual path still
                # preserves the token representation.
                token_output: torch.Tensor = torch.zeros_like(
                    x_flat[token_index]
                )

            else:
                token_output = torch.stack(
                    contributions,
                    dim=0,
                ).sum(dim=0)

            output_tokens.append(token_output)

        output_flat: torch.Tensor = torch.stack(
            output_tokens,
            dim=0,
        )

        output: torch.Tensor = output_flat.reshape(
            batch_size,
            sequence_length,
            embedding_dim,
        )

        routing_info: dict[str, torch.Tensor] = {
            "router_probabilities": (router_probabilities),
            "selected_experts": (topk_indices),
            "selected_weights": (topk_weights),
            "utilization": utilization,
            "accepted_route_mask": (accepted_route_mask),
        }

        return (
            output,
            balance_loss,
            routing_info,
        )

In [33]:
moe = SparseMoE(embedding_dim=8, hidden_dim=16, num_experts=4, top_k=2)

x: torch.Tensor = torch.randn(2, 3, 8)

(output, balance_loss, routing_info) = moe.forward_with_auxiliary(x)

print("input:", x.shape)

print("output:", output.shape)

print("balance loss:", balance_loss.item())

print("utilization:", routing_info["utilization"])

input: torch.Size([2, 3, 8])
output: torch.Size([2, 3, 8])
balance loss: 1.0037953853607178
utilization: tensor([0.2500, 0.3333, 0.2500, 0.1667])


In [34]:
plain_output: torch.Tensor = moe(x)

print(plain_output.shape)

torch.Size([2, 3, 8])


In [35]:
capacity_moe = SparseMoE(
    embedding_dim=8, hidden_dim=16, num_experts=4, top_k=2, capacity_factor=1.0
)

(capacity_output, capacity_balance_loss, capacity_info) = (
    capacity_moe.forward_with_auxiliary(x)
)

In [36]:
accepted_mask: torch.Tensor = capacity_info["accepted_route_mask"]

print("accepted routes:", accepted_mask)

print(
    "number accepted:",
    accepted_mask.sum().item(),
)

print(
    "number requested:",
    accepted_mask.numel(),
)

accepted routes: tensor([[ True,  True],
        [ True,  True],
        [ True, False],
        [ True,  True],
        [ True, False],
        [ True,  True]])
number accepted: 10
number requested: 12


In [37]:
dense_ffn = SwiGLU(embedding_dim=256, hidden_dim=768)
moe_ffn = SparseMoE(embedding_dim=256, hidden_dim=768, num_experts=8, top_k=2)

In [38]:
dense_parameters: int = count_parameters(dense_ffn)

moe_parameters: int = count_parameters(moe_ffn)

print(f"dense FFN total: {dense_parameters:,}")

print(f"MoE total: {moe_parameters:,}")

dense FFN total: 589,824
MoE total: 4,720,640


In [39]:
def moe_parameter_summary(
    *, embedding_dim: int, hidden_dim: int, num_experts: int, top_k: int
) -> dict[str, int]:
    """Compute simple dense/MoE parameter-count comparisons."""
    expert_parameters: int = 3 * embedding_dim * hidden_dim

    router_parameters: int = embedding_dim * num_experts

    total_moe_parameters: int = (
        num_experts * expert_parameters + router_parameters
    )

    active_moe_parameters: int = top_k * expert_parameters + router_parameters

    return {
        "dense_ffn": (expert_parameters),
        "moe_total": (total_moe_parameters),
        "moe_active_per_token": (active_moe_parameters),
    }


summary = moe_parameter_summary(
    embedding_dim=256, hidden_dim=768, num_experts=8, top_k=2
)

for name, value in summary.items():
    print(f"{name}: {value:,}")

dense_ffn: 589,824
moe_total: 4,720,640
moe_active_per_token: 1,181,696


### The Core MoE Decomposition

A sparse MoE can be understood through five components:

<pre>
experts
    ↓
what parameterized functions are available?

router
    ↓
how are experts scored?

selection
    ↓
which experts are active?

combination
    ↓
how are selected outputs weighted?

load management
    ↓
how are collapse and overload controlled?
</pre>

This decomposition is often more useful than memorizing individual MoE
model names.

## 8. Integrating MoE into a Transformer Block

A standard pre-norm Transformer block contains two residual updates:

$$
x'
=
x
+
\operatorname{Attention}
\left(
\operatorname{Norm}(x)
\right),
$$

followed by

$$
x''
=
x'
+
\operatorname{FFN}
\left(
\operatorname{Norm}(x')
\right).
$$

A sparse MoE Transformer does not need to change the attention path.

It replaces the dense feed-forward network:

$$
\operatorname{FFN}
$$

with

$$
\operatorname{SparseMoE}.
$$

Therefore:

<pre>
dense block

x
↓
attention
↓
residual
↓
SwiGLU
↓
residual


MoE block

x
↓
attention
↓
residual
↓
router
↓
selected experts
↓
weighted expert output
↓
residual
</pre>

In [40]:
class MoETransformerBlock(nn.Module):
    """Pre-norm Transformer block with a sparse MoE feed-forward path.

    Args:
        embedding_dim: Width of the residual stream.
        attention: Causal attention module.
        moe: Sparse mixture-of-experts feed-forward module.
    """

    def __init__(
        self, embedding_dim: int, attention: nn.Module, moe: SparseMoE
    ) -> None:
        super().__init__()

        self.attention_norm = nn.RMSNorm(embedding_dim)

        self.attention = attention
        self.feed_forward_norm = nn.RMSNorm(embedding_dim)
        self.moe = moe

    def forward(
        self, x: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, dict[str, torch.Tensor]]:
        """Apply one MoE Transformer block.

        Args:
            x: Residual-stream tensor with shape `(B, T, C)`.

        Returns:
            A tuple containing:
            - updated residual stream `(B, T, C)`,
            - scalar load-balancing loss,
            - routing diagnostics.
        """
        normalized: torch.Tensor = self.attention_norm(x)

        attention_output = self.attention(normalized)

        if not isinstance(attention_output, torch.Tensor):
            raise TypeError("Expected uncached attention output.")

        # First residual update: communication across token positions.
        x = x + attention_output

        normalized = self.feed_forward_norm(x)

        moe_output, balance_loss, routing_info = (
            self.moe.forward_with_auxiliary(normalized)
        )

        x = x + moe_output

        return x, balance_loss, routing_info


In [41]:
embedding_dim: int = 32

moe_block = MoETransformerBlock(
    embedding_dim=embedding_dim,
    attention=GroupedQueryAttention(
        embedding_dim=embedding_dim, num_query_heads=4, num_kv_heads=2
    ),
    moe=SparseMoE(
        embedding_dim=embedding_dim, hidden_dim=64, num_experts=4, top_k=2
    ),
)

x: torch.Tensor = torch.randn(2, 8, embedding_dim)

(block_output, balance_loss, routing_info) = moe_block(x)

print("input:", x.shape)

print("output:", block_output.shape)

print("balance loss:", balance_loss.item())

print("utilization:", routing_info["utilization"])

input: torch.Size([2, 8, 32])
output: torch.Size([2, 8, 32])
balance loss: 1.0115303993225098
utilization: tensor([0.2500, 0.2812, 0.2500, 0.2188])


In [42]:
class MoEGPT(nn.Module):
    """Decoder-only GPT whose feed-forward layers are sparse MoEs.

    Args:
        vocab_size: Vocabulary size.
        embedding_dim: Residual-stream width.
        num_query_heads: Number of query heads.
        num_kv_heads: Number of key/value heads.
        num_layers: Number of Transformer blocks.
        expert_hidden_dim: Hidden width of each expert.
        num_experts: Number of experts per MoE layer.
        top_k: Number of active experts per token.
    """

    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        num_query_heads: int,
        num_kv_heads: int,
        num_layers: int,
        expert_hidden_dim: int,
        num_experts: int,
        top_k: int,
    ) -> None:
        super().__init__()

        self.vocab_size: int = vocab_size
        self.embedding_dim: int = embedding_dim

        self.token_embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
        )

        self.blocks = nn.ModuleList(
            [
                MoETransformerBlock(
                    embedding_dim=embedding_dim,
                    attention=GroupedQueryAttention(
                        embedding_dim=embedding_dim,
                        num_query_heads=num_query_heads,
                        num_kv_heads=num_kv_heads,
                    ),
                    moe=SparseMoE(
                        embedding_dim=embedding_dim,
                        hidden_dim=expert_hidden_dim,
                        num_experts=num_experts,
                        top_k=top_k,
                    ),
                )
                for _ in range(num_layers)
            ]
        )

        self.final_norm = nn.RMSNorm(embedding_dim)

        self.lm_head = nn.Linear(
            embedding_dim,
            vocab_size,
            bias=False,
        )

    def forward(
        self,
        token_ids: torch.Tensor,
    ) -> tuple[
        torch.Tensor,
        torch.Tensor,
        list[dict[str, torch.Tensor]],
    ]:
        """Convert token IDs into logits and MoE auxiliary information.

        Args:
            token_ids: Token IDs with shape `(B, T)`.

        Returns:
            A tuple containing:
            - vocabulary logits `(B, T, V)`,
            - mean load-balancing loss,
            - routing diagnostics for every layer.
        """
        if token_ids.ndim != 2:
            raise ValueError("token_ids must have shape (B, T).")

        x: torch.Tensor = self.token_embedding(token_ids)

        balance_losses: list[torch.Tensor] = []

        routing_infos: list[dict[str, torch.Tensor]] = []

        for block in self.blocks:
            (x, balance_loss, routing_info) = block(x)

            balance_losses.append(balance_loss)

            routing_infos.append(routing_info)

        x = self.final_norm(x)

        logits: torch.Tensor = self.lm_head(x)

        mean_balance_loss: torch.Tensor = torch.stack(balance_losses).mean()

        return (logits, mean_balance_loss, routing_infos)

In [43]:
moe_model = MoEGPT(
    vocab_size=65,
    embedding_dim=64,
    num_query_heads=4,
    num_kv_heads=2,
    num_layers=3,
    expert_hidden_dim=128,
    num_experts=4,
    top_k=2,
)

token_ids: torch.Tensor = torch.randint(low=0, high=65, size=(2, 16))

(logits, balance_loss, routing_infos) = moe_model(token_ids)

print("logits:", logits.shape)

print("balance loss:", balance_loss.item())

print("routing layers:", len(routing_infos))

logits: torch.Size([2, 16, 65])
balance loss: 1.0167489051818848
routing layers: 3


### Total Parameters and Active Parameters

For an MoE model, total parameters and active parameters are different
quantities.

If every MoE layer contains $E$ experts but activates only $k$ experts
per token, then expert parameters scale approximately as

$$
P_{\text{expert,total}}
=
LEP_e,
$$

while the expert parameters used by one token scale as

$$
P_{\text{expert,active}}
=
LkP_e.
$$

The complete active-parameter count must also include the always-active
parts of the model:

- embeddings,
- attention,
- normalization,
- routers,
- and the output head.

Therefore, "active parameters" means parameters involved in one token's
forward computation, not the total parameter storage required by the
model.

## 9. A Subtle Point About Router Gradients

Sparse routing contains two different operations:

$$
\text{expert selection}
$$

and

$$
\text{expert weighting}.
$$

Top-k selection is discrete:

$$
\mathcal{E}(x)
=
\operatorname{TopK}(p(x), k).
$$

The selected routing weights can still be differentiable.

However, whether the task loss trains the router depends on how those
selected probabilities are used.

### Top-1 Is a Special Case

If top-1 routing renormalizes the selected probability, then

$$
g
=
\frac{p_e}{p_e}
=
1.
$$

The output becomes

$$
y=f_e(x),
$$

so the task loss has no gradient through the routing weight.

An alternative is to keep the raw selected probability:

$$
y=p_e f_e(x).
$$

Then the task loss can directly change the router's confidence.

This illustrates an important distinction:

<pre>
discrete expert identity
        ↓
not differentiable through ordinary backpropagation

continuous expert weight
        ↓
differentiable only if it actually affects the output
</pre>

### Where Router Gradients Come From

The router can receive learning signals from more than one source.

For a renormalized top-k MoE with $k>1$:

<pre>
language-model loss
        ↓
mixture output
        ↓
relative weights among selected experts
        ↓
selected router logits
</pre>

The balancing objective provides another path:

<pre>
load-balancing loss
        ↓
full router probabilities
        ↓
router logits
</pre>

The discrete top-k expert IDs themselves are not differentiated through
ordinary backpropagation.

In [44]:
from llmfp.nn import SparseMoE as ReusableSparseMoE

torch.manual_seed(7)

reusable_moe = ReusableSparseMoE(
    embedding_dim=16, hidden_dim=32, num_experts=4, top_k=2
)

test_input: torch.Tensor = torch.randn(2, 5, 16)

(test_output, test_balance_loss, test_routing_info) = (
    reusable_moe.forward_with_auxiliary(test_input)
)

print("input:", test_input.shape)

print("output:", test_output.shape)

print("selected experts:", test_routing_info["selected_experts"].shape)

print("selected weights:", test_routing_info["selected_weights"].shape)

print("utilization:", test_routing_info["utilization"])

print("balance loss:", test_balance_loss.item())

assert test_output.shape == (2, 5, 16)

assert test_routing_info["selected_experts"].shape == (10, 2)

assert test_routing_info["selected_weights"].shape == (10, 2)

assert torch.allclose(
    test_routing_info["selected_weights"].sum(dim=-1),
    torch.ones(10, device=test_input.device),
)

assert torch.allclose(
    test_routing_info["utilization"].sum(),
    torch.tensor(1.0, device=test_input.device),
)

assert torch.isfinite(test_balance_loss)

input: torch.Size([2, 5, 16])
output: torch.Size([2, 5, 16])
selected experts: torch.Size([10, 2])
selected weights: torch.Size([10, 2])
utilization: tensor([0.2500, 0.2500, 0.2500, 0.2500])
balance loss: 1.0


In [45]:
reusable_moe.zero_grad()

(output, balance_loss, routing_info) = reusable_moe.forward_with_auxiliary(
    test_input
)

task_loss: torch.Tensor = output.square().mean()

total_loss: torch.Tensor = task_loss + 0.01 * balance_loss

total_loss.backward()

In [46]:
router_gradient = reusable_moe.router.weight.grad

assert router_gradient is not None

print(
    "router gradient norm:",
    torch.linalg.vector_norm(router_gradient).item(),
)

router gradient norm: 0.004306045360863209


In [47]:
for expert_index, expert in enumerate(reusable_moe.experts):
    first_parameter = next(expert.parameters())

    gradient = first_parameter.grad

    if gradient is None:
        print(f"expert {expert_index}: no task gradient")
    else:
        print(
            f"expert {expert_index}: "
            f"{torch.linalg.vector_norm(gradient).item():.6f}"
        )

expert 0: 0.005009
expert 1: 0.001485
expert 2: 0.003939
expert 3: 0.003460


In [48]:
top1_normalized = ReusableSparseMoE(
    embedding_dim=16,
    hidden_dim=32,
    num_experts=4,
    top_k=1,
    renormalize_selected_weights=True,
)
top1_normalized.zero_grad()

output, _, _ = top1_normalized.forward_with_auxiliary(test_input)

task_loss = output.square().mean()

task_loss.backward()

normalized_router_grad = top1_normalized.router.weight.grad

print(normalized_router_grad)

tensor([[-1.4929e-10,  1.5966e-10, -2.2327e-10,  4.3830e-11,  7.9422e-11,
         -6.5770e-11,  5.7578e-11, -5.9890e-11,  3.6952e-11,  1.8716e-10,
         -1.0918e-10, -4.3246e-11,  1.4576e-10,  4.0947e-11, -1.2426e-10,
         -5.9360e-11],
        [ 7.9120e-11, -3.7070e-11,  3.2082e-11,  1.8067e-11,  2.7172e-12,
          1.7480e-11, -1.0946e-11, -1.8096e-11,  7.6036e-13, -3.2683e-11,
          2.4338e-11,  1.8617e-11, -2.2456e-11, -2.1077e-12,  4.8482e-11,
          3.1583e-11],
        [ 3.8608e-11, -6.4379e-11,  9.9618e-11, -3.1387e-11, -4.2297e-11,
          2.5447e-11, -2.4392e-11,  3.9687e-11, -1.9435e-11, -8.0693e-11,
          4.4516e-11,  1.3273e-11, -6.4300e-11, -2.0103e-11,  4.0570e-11,
          1.5292e-11],
        [ 3.1560e-11, -5.8215e-11,  9.1569e-11, -3.0510e-11, -3.9842e-11,
          2.2844e-11, -2.2239e-11,  3.8299e-11, -1.8278e-11, -7.3778e-11,
          4.0330e-11,  1.1356e-11, -5.9001e-11, -1.8736e-11,  3.5205e-11,
          1.2486e-11]])


In [49]:
top1_raw = ReusableSparseMoE(
    embedding_dim=16,
    hidden_dim=32,
    num_experts=4,
    top_k=1,
    renormalize_selected_weights=False,
)

top1_raw.zero_grad()

output, _, _ = top1_raw.forward_with_auxiliary(test_input)

task_loss = output.square().mean()

task_loss.backward()

raw_router_grad = top1_raw.router.weight.grad

print(
    "raw top-1 router gradient norm:",
    torch.linalg.vector_norm(raw_router_grad).item(),
)

raw top-1 router gradient norm: 0.001328898360952735


## 10. Dense FFN vs Sparse MoE

| Property | Dense FFN | Sparse MoE |
|---|---|---|
| Available FFNs | 1 | $E$ experts |
| Active FFNs per token | 1 | $k$ |
| Routing | None | Learned router |
| Token-dependent parameters | No | Yes |
| Total expert parameters | $P_e$ | $E P_e$ |
| Active expert parameters | $P_e$ | $k P_e$ |
| Dispatch / gather | No | Yes |
| Load imbalance | No | Possible |
| Capacity constraints | No | Often needed |
| Auxiliary routing loss | No | Often used |
| Expert specialization | Not applicable | Can emerge during training |

## Takeaways

A dense Transformer feed-forward network applies the same parameters to
every token.

Mixture of Experts replaces that assumption with learned conditional
computation.

For each token representation, a router produces expert scores:

$$
r = W_r x,
$$

followed by routing probabilities:

$$
p(e\mid x)
=
\operatorname{softmax}(r)_e.
$$

Sparse top-k routing selects only a small subset of experts:

$$
\mathcal{E}(x)
=
\operatorname{TopK}(p(x), k).
$$

The token update is constructed from the selected expert outputs:

$$
y
=
\sum_{e\in\mathcal{E}(x)}
g_e f_e(x).
$$

This separates two scaling quantities:

$$
\text{total expert capacity}
\propto E,
$$

while

$$
\text{expert computation per token}
\propto k.
$$

Routing also determines which experts receive task gradients, so sparse
MoE introduces a coupled learning and resource-allocation problem.

Load balancing provides an optimization incentive for distributing work,
while expert capacity imposes a hard execution constraint.

Finally, sparse routing combines discrete and continuous operations:

<pre>
top-k expert identity
        ↓
discrete selection

routing weights
        ↓
continuous differentiation

selected experts
        ↓
conditional computation
</pre>

The central MoE idea is therefore not merely sparsity.

It is:

$$
\boxed{
\text{learned input-dependent parameter selection}
}
$$